# Camada Silver - limpeza e padronização

Entrada: `bronze.telecom.*` — Saída: `silver.telecom.*`

## Responsabilidades

1. **Recorte** — RS, período de referência único
2. **Tipagem** — texto para inteiro e decimal
3. **Valores especiais do SIDRA** — `-`, `..`, `...`, `X`
4. **Separador decimal** — vírgula para ponto
5. **Redução de grão** — descarte da velocidade exata
6. **Normalização** — separação de atributos com dependência funcional
7. **Qualidade** — completude, consistência, unicidade, acurácia, outliers

## Convenção de nomes

A unidade de medida faz parte do nome da coluna:

| Sufixo | Unidade |
|---|---|
| `_qtd` | contagem de unidades |
| `_hab` | habitantes |
| `_pct` | percentual (0 a 100) |
| `_brl` | reais |
| `_mil_brl` | mil reais (unidade original do IBGE) |
| `_km2` | quilômetros quadrados |

## Dependências funcionais verificadas

Testado sobre o recorte do RS:

- `cnpj` **determina** `empresa`, `grupo_economico` e `porte` — zero inconsistências
- `codigo_ibge` **determina** `municipio` e `uf`
- `tecnologia` **não determina** `meio_acesso` — hipótese inicial refutada pelo dado

A última merece atenção. A suposição natural seria que cada tecnologia usa um meio
físico fixo, mas ETHERNET aparece tanto sobre fibra quanto sobre cabo metálico. Por
isso `meio_acesso` **permanece no grão** da tabela de acessos, e a tabela de
tecnologias tem **chave composta**. Tratar a relação como funcional duplicaria linhas
no join da camada Gold e inflaria os acessos.

## Dois denominadores de mercado

O Censo separa domicílios por espécie. A distinção importa para telecom:

| Espécie | Código SIDRA | Tem contrato de banda larga? |
|---|---|---|
| Particular permanente ocupado | 59998 | sim, é o caso típico |
| Não ocupado — uso ocasional | 60003 | **frequentemente sim** — casa de praia com câmera, streaming, trabalho remoto |
| Não ocupado — vago | 60002 | raramente |
| Total recenseado | 59993 | universo completo |

Usar apenas domicílios ocupados como denominador distorce municípios de veraneio: o
acesso existe, mas o imóvel não é contado. Por isso a camada Silver traz **as duas
contagens**, e a Gold calcula penetração sobre ambas.

## 1. Parâmetros

In [0]:
CATALOGO_BRONZE = "bronze"
CATALOGO_SILVER = "silver"
SCHEMA          = "telecom"

UF_ALVO = "RS"
ANO_REF = "2026"
MES_REF = "7"

QTD_MUNICIPIOS_RS = 497
PERIODO_REF = f"{ANO_REF}-{int(MES_REF):02d}"

print(f"Recorte: {UF_ALVO}, período {PERIODO_REF}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql import Row
from datetime import datetime, timezone
import requests

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO_SILVER}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO_SILVER}.{SCHEMA}")
print(f"Destino: {CATALOGO_SILVER}.{SCHEMA}")

## 2. Funções auxiliares

### 2.1 Tratamento de valores especiais

O SIDRA usa códigos textuais para ausência de dado. Converter direto para número
transformaria todos em nulo, sem registro do motivo.

| Código | Significado |
|---|---|
| `-` | zero absoluto |
| `..` | não se aplica |
| `...` | dado não disponível |
| `X` | suprimido por sigilo |

In [0]:
VALORES_ESPECIAIS_SIDRA = ["-", "..", "...", "X", ""]

def limpar_numero(coluna, decimal_virgula: bool = False):
    """Converte texto para número, tratando valores especiais e vírgula decimal."""
    col = F.trim(F.col(coluna))
    col = F.when(col.isin(VALORES_ESPECIAIS_SIDRA), None).otherwise(col)
    if decimal_virgula:
        col = F.regexp_replace(col, r"\.", "")   # separador de milhar
        col = F.regexp_replace(col, ",", ".")    # vírgula decimal
    return col


def normalizar_nome_coluna(nome: str) -> str:
    import re
    acentos = str.maketrans(
        "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
        "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC")
    n = nome.strip().translate(acentos).lower()
    n = re.sub(r"[^a-z0-9]+", "_", n)
    return re.sub(r"_+", "_", n).strip("_")

### 2.2 Formatação para exibição

`br()` converte para o padrão brasileiro e aceita sufixo de unidade. A formatação é
**apenas visual** — as tabelas persistidas mantêm os tipos numéricos, para não
inviabilizar ordenação e agregação.

In [0]:
def br(coluna, decimais: int = 0, sufixo: str = ""):
    """Formata número no padrão brasileiro: 1.474 / 26,46 / 26,46 %"""
    txt = F.format_number(F.col(coluna), decimais)   # 1,474.00 (padrão US)
    txt = F.translate(txt, ".,", ",.")               # troca simultânea
    return F.concat(txt, F.lit(sufixo)) if sufixo else txt


def exibir(df, numericos: dict, texto: list = None):
    """numericos: {"coluna": (casas_decimais, sufixo)}"""
    texto = texto or []
    sel  = [F.col(x) for x in texto]
    sel += [br(col, d, s).alias(col) for col, (d, s) in numericos.items()]
    return df.select(*sel)

### 2.3 Gravação e documentação

`documentar_colunas` aplica `COMMENT ON COLUMN` em lote. Os comentários ficam
visíveis no Catalog Explorer e constituem o catálogo de dados exigido pelo trabalho:
descrição, unidade, domínio de valores e linhagem de cada campo.

In [0]:
def gravar_silver(df, tabela: str, comentario: str):
    df = df.withColumn("_data_processamento",
                       F.lit(datetime.now(timezone.utc).isoformat()))
    nome = f"{CATALOGO_SILVER}.{SCHEMA}.{tabela}"
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(nome))
    spark.sql(f"COMMENT ON TABLE {nome} IS '{comentario.replace(chr(39), '')}'")
    print(f"{nome}: {spark.table(nome).count():,} linhas / {len(df.columns)} colunas")
    return spark.table(nome)


def documentar_colunas(tabela: str, dicionario: dict):
    nome = f"{CATALOGO_SILVER}.{SCHEMA}.{tabela}"
    for coluna, descricao in dicionario.items():
        spark.sql(f"COMMENT ON COLUMN {nome}.{coluna} IS "
                  f"'{descricao.replace(chr(39), '')}'")
    print(f"{nome}: {len(dicionario)} colunas documentadas")

### 2.4 Pivot do SIDRA

A API retorna formato longo: uma linha por município por variável — ou, quando há
classificação, uma linha por município por categoria.

A função pivota para uma linha por município. O parâmetro `coluna_pivot` indica se o
desdobramento é por variável (`variavel_codigo`) ou por categoria de classificação
(por exemplo `especie_codigo`, na tabela 4711).

In [0]:
def pivotar_sidra(tabela_bronze: str, mapa: dict,
                  coluna_pivot: str = "variavel_codigo"):
    """mapa: {"93": "populacao_residente_hab", ...} ou {"59993": "domicilios_total_qtd"}"""
    df = spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.{tabela_bronze}")

    col_mun = next(x for x in df.columns if "municipio_codigo" in x)
    col_piv = next(x for x in df.columns if coluna_pivot in x)

    pivotado = (
        df.select(
            F.col(col_mun).alias("codigo_ibge"),
            F.col(col_piv).alias("chave_pivot"),
            limpar_numero("valor").cast(DoubleType()).alias("valor"),
        )
        .groupBy("codigo_ibge")
        .pivot("chave_pivot", list(mapa.keys()))
        .agg(F.first("valor"))
    )
    for cod, nome in mapa.items():
        pivotado = pivotado.withColumnRenamed(cod, nome)
    return pivotado

### 2.5 Ingestão complementar — domicílios por espécie

A tabela 4711 do SIDRA não estava prevista na camada Bronze original. Como a
consulta usa **classificação** (`c3` = Espécie), a URL tem um segmento a mais do que
as consultas anteriores.

A célula abaixo faz a ingestão e grava na Bronze, mantendo o princípio da camada:
dado como veio, tudo como texto.

In [0]:
SIDRA_ESPECIE = {
    "tabela": 4711,
    "variaveis": "617",
    "periodo": "2022",
    "classificacao": "3",
    "categorias": "59993,59998,60001,60002,60003",
}

def baixar_sidra_classificado(tabela, variaveis, periodo,
                              classificacao=None, categorias=None,
                              uf_codigo="43"):
    """Consulta o SIDRA no nível município, com classificação opcional."""
    url = (f"https://apisidra.ibge.gov.br/values"
           f"/t/{tabela}/n6/in%20n3%20{uf_codigo}"
           f"/v/{variaveis}/p/{periodo}")
    if classificacao and categorias:
        url += f"/c{classificacao}/{categorias}"

    r = requests.get(url, timeout=300)
    r.raise_for_status()
    dados = r.json()

    cabecalho, linhas = dados[0], dados[1:]
    colunas = {k: normalizar_nome_coluna(v) for k, v in cabecalho.items()}
    registros = [{colunas[k]: str(v) for k, v in linha.items()} for linha in linhas]

    from pyspark.sql.types import StringType, StructType, StructField
    schema = StructType([StructField(x, StringType(), True) for x in registros[0]])
    print(f"Tabela {tabela}: {len(registros)} registros")
    return spark.createDataFrame(registros, schema=schema), url

In [0]:
df_especie, url_especie = baixar_sidra_classificado(**SIDRA_ESPECIE)

df_especie = (
    df_especie
    .withColumn("_origem",        F.lit(url_especie))
    .withColumn("_fonte",         F.lit("IBGE - API SIDRA - tabela 4711 - Censo 2022"))
    .withColumn("_data_ingestao", F.lit(datetime.now(timezone.utc).isoformat()))
)

nome_bronze = f"{CATALOGO_BRONZE}.{SCHEMA}.ibge_domicilios_especie_raw"
(df_especie.write.mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(nome_bronze))

spark.sql(f"COMMENT ON TABLE {nome_bronze} IS "
    "'Camada Bronze. Domicilios recenseados por especie, por municipio do RS, Censo "
    "2022. Categorias: 59993 Total, 59998 Particular permanente ocupado, 60001 Nao "
    "ocupado, 60002 Nao ocupado vago, 60003 Nao ocupado uso ocasional. Fonte: API "
    "SIDRA do IBGE, tabela 4711, variavel 617, classificacao 3.'")

print(f"{nome_bronze}: {spark.table(nome_bronze).count():,} linhas")
print(f"Colunas: {spark.table(nome_bronze).columns}")

Conferência das colunas das demais tabelas do SIDRA, antes de pivotar.

In [0]:
for t in ["ibge_populacao_raw", "ibge_domicilios_raw", "ibge_pib_raw"]:
    print(f"{t}:\n  {spark.table(f'{CATALOGO_BRONZE}.{SCHEMA}.{t}').columns}\n")

## 3. Acessos

Recorte, tipagem e redução de grão.

**Grão de saída:** município × CNPJ × tecnologia × meio de acesso × faixa de
velocidade × tipo de pessoa × tipo de produto

A coluna `velocidade` (valor exato, 1.135 valores distintos no RS) é descartada.
`faixa_velocidade`, com 5 categorias, responde às perguntas de negócio com
cardinalidade muito menor.

**Nenhum tipo de acesso é descartado.** Pessoa física e jurídica, INTERNET, linha
dedicada e M2M permanecem na tabela. A camada Gold expõe cada recorte separadamente,
já que toda conexão ativa é mercado atendido.

In [0]:
bronze_acessos = spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.anatel_acessos_raw")

acessos_recorte = bronze_acessos.filter(
    (F.col("uf") == UF_ALVO) &
    (F.col("ano") == ANO_REF) &
    (F.col("mes") == MES_REF)
)

print(f"Linhas no recorte {UF_ALVO} {PERIODO_REF}: {acessos_recorte.count():,}")

In [0]:
GRAO_ACESSOS = ["periodo", "codigo_ibge", "cnpj", "tecnologia", "meio_acesso",
                "faixa_velocidade", "tipo_pessoa", "tipo_produto"]

silver_acessos = (
    acessos_recorte
    .select(
        F.lit(PERIODO_REF).alias("periodo"),
        F.col("codigo_ibge_municipio").alias("codigo_ibge"),
        F.col("cnpj"),
        F.col("tecnologia"),
        F.col("meio_de_acesso").alias("meio_acesso"),
        F.col("faixa_de_velocidade").alias("faixa_velocidade"),
        F.col("tipo_de_pessoa").alias("tipo_pessoa"),
        F.col("tipo_de_produto").alias("tipo_produto"),
        F.col("acessos").cast(IntegerType()).alias("acessos_qtd"),
    )
    .groupBy(*GRAO_ACESSOS)
    .agg(F.sum("acessos_qtd").alias("acessos_qtd"))
)

gravar_silver(silver_acessos, "acessos",
    "Camada Silver. Acessos de banda larga fixa no RS, periodo de referencia unico. "
    "Grao: municipio x CNPJ x tecnologia x meio de acesso x faixa de velocidade x "
    "tipo de pessoa x tipo de produto. A coluna velocidade exata foi descartada por "
    "cardinalidade (1135 valores distintos) sem ganho analitico. O meio de acesso "
    "permanece no grao porque nao e determinado funcionalmente pela tecnologia. "
    "Nenhum tipo de acesso e descartado.")

documentar_colunas("acessos", {
    "periodo": "Periodo de referencia no formato AAAA-MM. Valor unico neste MVP: "
        "2026-07. Linhagem: colunas ano e mes de bronze.anatel_acessos_raw.",
    "codigo_ibge": "Codigo IBGE do municipio, 7 digitos. Dominio: municipios do RS "
        "(4300034 a 4323804). Linhagem: campo Codigo IBGE Municipio da Anatel.",
    "cnpj": "CNPJ da prestadora, 14 digitos sem formatacao. Chave de identidade "
        "competitiva. Linhagem: campo CNPJ da Anatel.",
    "tecnologia": "Tecnologia de acesso declarada. Dominio: 24 categorias, entre elas "
        "FTTH, ETHERNET, HFC, VSAT, ADSL2. Linhagem: campo Tecnologia da Anatel.",
    "meio_acesso": "Meio fisico de transmissao. Dominio: Fibra, Radio, Satelite, "
        "Cabo Metalico, Cabo Coaxial. Nao e determinado pela tecnologia. "
        "Linhagem: campo Meio de Acesso da Anatel.",
    "faixa_velocidade": "Faixa de velocidade contratada. Dominio: 0Kbps a 512Kbps, "
        "512kbps a 2Mbps, 2Mbps a 12Mbps, 12Mbps a 34Mbps, maior que 34Mbps. "
        "Linhagem: campo Faixa de Velocidade da Anatel.",
    "tipo_pessoa": "Natureza do assinante. Dominio: Pessoa Fisica, Pessoa Juridica. "
        "Linhagem: campo Tipo de Pessoa da Anatel.",
    "tipo_produto": "Tipo de produto contratado. Dominio: INTERNET, LINHA_DEDICADA, "
        "M2M, OUTROS. Linhagem: campo Tipo de Produto da Anatel.",
    "acessos_qtd": "Quantidade de acessos ativos. Unidade: acessos. Medida aditiva. "
        "Dominio: inteiro maior ou igual a 1. Linhagem: soma do campo Acessos da "
        "Anatel, agregado pelo grao desta tabela.",
})

In [0]:
sa = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.acessos")
sa.printSchema()

total_geral = sa.agg(F.sum("acessos_qtd")).first()[0]
print(f"Total de acessos no RS em {PERIODO_REF}: {total_geral:,}")

display(exibir(sa.limit(20),
    numericos={"acessos_qtd": (0, "")},
    texto=["periodo", "codigo_ibge", "cnpj", "tecnologia", "meio_acesso",
           "faixa_velocidade", "tipo_pessoa", "tipo_produto"]))

In [0]:
por_faixa = (
    sa.groupBy("faixa_velocidade")
      .agg(F.sum("acessos_qtd").alias("acessos_qtd"),
           F.count("*").alias("linhas_qtd"))
      .withColumn("participacao_pct",
          F.round(100 * F.col("acessos_qtd") / F.lit(total_geral), 2))
      .orderBy(F.col("acessos_qtd").desc())
)

display(exibir(por_faixa,
    numericos={"acessos_qtd": (0, ""), "linhas_qtd": (0, ""),
               "participacao_pct": (2, " %")},
    texto=["faixa_velocidade"]))

In [0]:
por_segmento = (
    sa.groupBy("tipo_pessoa", "tipo_produto")
      .agg(F.sum("acessos_qtd").alias("acessos_qtd"))
      .withColumn("participacao_pct",
          F.round(100 * F.col("acessos_qtd") / F.lit(total_geral), 2))
      .orderBy(F.col("acessos_qtd").desc())
)

display(exibir(por_segmento,
    numericos={"acessos_qtd": (0, ""), "participacao_pct": (2, " %")},
    texto=["tipo_pessoa", "tipo_produto"]))

In [0]:
por_meio = (
    sa.groupBy("meio_acesso")
      .agg(F.sum("acessos_qtd").alias("acessos_qtd"),
           F.countDistinct("tecnologia").alias("tecnologias_qtd"))
      .withColumn("participacao_pct",
          F.round(100 * F.col("acessos_qtd") / F.lit(total_geral), 2))
      .orderBy(F.col("acessos_qtd").desc())
)

display(exibir(por_meio,
    numericos={"acessos_qtd": (0, ""), "tecnologias_qtd": (0, ""),
               "participacao_pct": (2, " %")},
    texto=["meio_acesso"]))

## 4. Empresas

`cnpj` determina `empresa`, `grupo_economico` e `porte` — verificado, zero
inconsistências.

**Observação central para a análise competitiva:** a Anatel classifica todos os
provedores regionais como grupo econômico `OUTROS`, o que corresponde a 55% das
linhas do RS. Por isso a identidade do concorrente neste projeto é o **CNPJ**.

In [0]:
silver_empresas = (
    acessos_recorte
    .select(F.col("cnpj"), F.col("empresa"), F.col("grupo_economico"),
            F.col("porte_da_prestadora").alias("porte"))
    .distinct()
    .withColumn("grupo_identificado", F.col("grupo_economico") != "OUTROS")
)

gravar_silver(silver_empresas, "empresas",
    "Camada Silver. Prestadoras de SCM com atuacao no RS no periodo de referencia. "
    "Chave: CNPJ. O campo grupo_economico traz OUTROS para todos os provedores "
    "regionais, o que o inviabiliza como chave de analise competitiva.")

documentar_colunas("empresas", {
    "cnpj": "CNPJ da prestadora, 14 digitos sem formatacao. Chave primaria. "
        "Linhagem: campo CNPJ da Anatel.",
    "empresa": "Nome comercial da prestadora. Determinado funcionalmente pelo CNPJ.",
    "grupo_economico": "Grupo economico declarado. Dominio: 16 categorias no RS. "
        "OUTROS agrega todos os provedores regionais, 55 por cento das linhas.",
    "porte": "Porte segundo a Anatel. Dominio: Pequeno Porte, Grande Porte.",
    "grupo_identificado": "Indica grupo diferente de OUTROS. Dominio: true, false.",
})

In [0]:
emp = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.empresas")

print(f"Prestadoras no RS: {emp.count()}")
print(f"  com grupo identificado: {emp.filter('grupo_identificado').count()}")
print(f"  classificadas como OUTROS: {emp.filter('not grupo_identificado').count()}")

ranking = (
    sa.groupBy("cnpj")
      .agg(F.sum("acessos_qtd").alias("acessos_qtd"),
           F.countDistinct("codigo_ibge").alias("municipios_qtd"))
      .join(emp, "cnpj")
      .withColumn("share_estadual_pct",
          F.round(100 * F.col("acessos_qtd") / F.lit(total_geral), 3))
      .orderBy(F.col("acessos_qtd").desc()).limit(25)
)

display(exibir(ranking,
    numericos={"acessos_qtd": (0, ""), "municipios_qtd": (0, ""),
               "share_estadual_pct": (3, " %")},
    texto=["empresa", "grupo_economico", "porte"]))

## 5. Tecnologias

**Chave composta:** `tecnologia` + `meio_acesso`.

A hipótese inicial era que cada tecnologia usaria um meio físico fixo. O dado
refutou: ETHERNET aparece sobre fibra e sobre cabo metálico. Tratar a relação como
funcional produziria chave duplicada na dimensão e inflaria os acessos no join da
camada Gold.

In [0]:
silver_tecnologias = (
    acessos_recorte
    .select(F.col("tecnologia"), F.col("meio_de_acesso").alias("meio_acesso"))
    .distinct()
    .withColumn("e_fibra", F.col("meio_acesso") == "Fibra")
)

gravar_silver(silver_tecnologias, "tecnologias",
    "Camada Silver. Combinacoes de tecnologia e meio fisico presentes no RS. Chave "
    "composta: tecnologia + meio_acesso. A tecnologia NAO determina funcionalmente o "
    "meio de acesso: ETHERNET, por exemplo, aparece sobre fibra e sobre cabo "
    "metalico. A flag e_fibra sustenta as analises de penetracao de fibra.")

documentar_colunas("tecnologias", {
    "tecnologia": "Tecnologia de acesso declarada. Parte da chave composta. "
        "Dominio: 24 categorias. Linhagem: campo Tecnologia da Anatel.",
    "meio_acesso": "Meio fisico de transmissao. Parte da chave composta. Dominio: "
        "Fibra, Radio, Satelite, Cabo Metalico, Cabo Coaxial.",
    "e_fibra": "Indica meio de acesso igual a Fibra. Dominio: true, false.",
})

Evidência da relação não funcional entre tecnologia e meio de acesso

In [0]:
tec = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.tecnologias")

multiplas = (
    tec.groupBy("tecnologia")
       .agg(F.count("*").alias("meios_qtd"),
            F.concat_ws(", ", F.collect_list("meio_acesso")).alias("meios"))
       .filter(F.col("meios_qtd") > 1)
       .orderBy(F.col("meios_qtd").desc())
)

print(f"Combinacoes tecnologia x meio: {tec.count()}")
print(f"Tecnologias distintas: {tec.select('tecnologia').distinct().count()}")
print(f"Tecnologias com mais de um meio: {multiplas.count()}")

display(multiplas)

## 6. Municípios

Consolida cinco fontes do IBGE em uma linha por município: hierarquia geográfica,
população, domicílios por espécie, domicílios ocupados com moradores, e PIB.

O **setor econômico dominante** é definido pelo maior VAB setorial — critério
objetivo, sem julgamento subjetivo.

In [0]:
localidades = (
    spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.ibge_municipios_raw")
    .select("codigo_ibge", "nome_municipio",
            "microrregiao_nome", "mesorregiao_nome",
            "regiao_imediata_nome", "regiao_intermediaria_nome", "uf_sigla")
)

populacao = pivotar_sidra("ibge_populacao_raw", {
    "93":   "populacao_residente_hab",
    "6318": "area_km2",
    "614":  "densidade_demografica_hab_km2",
})

domicilios = pivotar_sidra("ibge_domicilios_raw", {
    "381":  "domicilios_ocupados_qtd",
    "382":  "moradores_em_domicilios_qtd",
    "5930": "media_moradores_domicilio",
})

# Tabela 4711: pivot pela categoria de especie, nao pela variavel
especie = pivotar_sidra("ibge_domicilios_especie_raw", {
    "59993": "domicilios_total_qtd",
    "59998": "domicilios_ocupados_esp_qtd",
    "60001": "domicilios_nao_ocupados_qtd",
    "60002": "domicilios_vagos_qtd",
    "60003": "domicilios_uso_ocasional_qtd",
}, coluna_pivot="especie_codigo")

pib = pivotar_sidra("ibge_pib_raw", {
    "37":   "pib_total_mil_brl",
    "543":  "impostos_liquidos_mil_brl",
    "498":  "vab_total_mil_brl",
    "513":  "vab_agropecuaria_mil_brl",
    "517":  "vab_industria_mil_brl",
    "6575": "vab_servicos_mil_brl",
    "525":  "vab_administracao_mil_brl",
})

print(f"localidades={localidades.count()} | populacao={populacao.count()} | "
      f"domicilios={domicilios.count()} | especie={especie.count()} | pib={pib.count()}")

In [0]:
silver_municipios = (
    localidades
    .join(populacao,  "codigo_ibge", "left")
    .join(domicilios, "codigo_ibge", "left")
    .join(especie,    "codigo_ibge", "left")
    .join(pib,        "codigo_ibge", "left")
    # Setor economico dominante: maior VAB setorial.
    # VAB nulo significa ausencia do setor, nao dado desconhecido - por isso o
    # coalesce para zero. Sem ele, F.greatest retorna nulo se qualquer argumento
    # for nulo, e o setor dominante ficaria nulo em todo o estado.
    .withColumn("vab_agro_c", F.coalesce("vab_agropecuaria_mil_brl",  F.lit(0.0)))
    .withColumn("vab_ind_c",  F.coalesce("vab_industria_mil_brl",     F.lit(0.0)))
    .withColumn("vab_serv_c", F.coalesce("vab_servicos_mil_brl",      F.lit(0.0)))
    .withColumn("vab_adm_c",  F.coalesce("vab_administracao_mil_brl", F.lit(0.0)))
    .withColumn("vab_maior",
        F.greatest("vab_agro_c", "vab_ind_c", "vab_serv_c", "vab_adm_c"))
    # Ordem dos testes: servicos primeiro, agropecuaria por ultimo. Em caso de
    # empate em zero (municipio sem VAB informado), a ordem anterior classificaria
    # tudo como Agropecuaria.
    .withColumn("setor_dominante",
        F.when(F.col("vab_maior") <= 0, None)
         .when(F.col("vab_serv_c") == F.col("vab_maior"), "Servicos")
         .when(F.col("vab_adm_c")  == F.col("vab_maior"), "Administracao publica")
         .when(F.col("vab_ind_c")  == F.col("vab_maior"), "Industria")
         .when(F.col("vab_agro_c") == F.col("vab_maior"), "Agropecuaria"))
    .withColumn("pib_per_capita_brl",
        F.round(F.col("pib_total_mil_brl") * 1000 / F.col("populacao_residente_hab"), 2))
    .withColumn("porte_populacional",
        F.when(F.col("populacao_residente_hab") <   5000, "Ate 5 mil")
         .when(F.col("populacao_residente_hab") <  20000, "5 a 20 mil")
         .when(F.col("populacao_residente_hab") < 100000, "20 a 100 mil")
         .otherwise("Acima de 100 mil"))
    # participacao de imoveis de uso ocasional no total recenseado
    .withColumn("uso_ocasional_pct",
        F.round(100 * F.col("domicilios_uso_ocasional_qtd")
                / F.col("domicilios_total_qtd"), 2))
    .withColumn("perfil_ocupacao",
        F.when(F.col("uso_ocasional_pct") >= 30, "Veraneio ou turismo")
         .when(F.col("uso_ocasional_pct") >= 10, "Ocupacao mista")
         .otherwise("Ocupacao permanente"))
    .drop("vab_maior", "vab_agro_c", "vab_ind_c", "vab_serv_c", "vab_adm_c")
)

gravar_silver(silver_municipios, "municipios",
    "Camada Silver. Municipios do RS consolidando cinco fontes do IBGE: hierarquia "
    "geografica (API Localidades), populacao e area (SIDRA 4714), domicilios "
    "ocupados e moradores (SIDRA 4712), domicilios recenseados por especie "
    "(SIDRA 4711) e PIB com VAB setorial (SIDRA 5938). Traz DOIS denominadores de "
    "mercado: domicilios_ocupados_qtd (padrao IBGE) e domicilios_total_qtd (universo "
    "recenseado, inclui uso ocasional e vagos). Valores do IBGE em mil reais.")

In [0]:
documentar_colunas("municipios", {
    "codigo_ibge": "Codigo IBGE do municipio, 7 digitos. Chave primaria. "
        "Dominio: 4300034 a 4323804. Linhagem: IBGE API Localidades.",
    "nome_municipio": "Nome oficial do municipio. Linhagem: IBGE API Localidades.",
    "microrregiao_nome": "Microrregiao geografica do IBGE. Linhagem: API Localidades.",
    "mesorregiao_nome": "Mesorregiao geografica do IBGE. Dominio: 7 categorias no RS.",
    "regiao_imediata_nome": "Regiao geografica imediata, divisao vigente.",
    "regiao_intermediaria_nome": "Regiao geografica intermediaria do IBGE.",
    "uf_sigla": "Sigla da unidade federativa. Valor unico neste MVP: RS.",
    "populacao_residente_hab": "Populacao residente. Unidade: habitantes. "
        "Linhagem: SIDRA 4714 v/93, Censo 2022.",
    "area_km2": "Area da unidade territorial. Unidade: quilometros quadrados. "
        "Linhagem: SIDRA 4714 v/6318.",
    "densidade_demografica_hab_km2": "Densidade demografica. Unidade: habitantes por "
        "quilometro quadrado. Linhagem: SIDRA 4714 v/614.",
    "domicilios_ocupados_qtd": "Domicilios particulares permanentes ocupados. "
        "Unidade: domicilios. Denominador conservador de mercado. Nao inclui imoveis "
        "de uso ocasional nem vagos. Linhagem: SIDRA 4712 v/381.",
    "moradores_em_domicilios_qtd": "Moradores em domicilios particulares permanentes "
        "ocupados. Unidade: pessoas. Linhagem: SIDRA 4712 v/382.",
    "media_moradores_domicilio": "Media de moradores por domicilio. Unidade: "
        "moradores por domicilio. Linhagem: SIDRA 4712 v/5930.",
    "domicilios_total_qtd": "Total de domicilios recenseados, todas as especies. "
        "Unidade: domicilios. Denominador de mercado enderecavel, ja que qualquer "
        "imovel e potencial contratante de banda larga. "
        "Linhagem: SIDRA 4711 v/617 c3/59993, Censo 2022.",
    "domicilios_ocupados_esp_qtd": "Domicilios particulares permanentes ocupados "
        "segundo a tabela 4711. Unidade: domicilios. Usado para validacao cruzada "
        "com domicilios_ocupados_qtd, que vem da tabela 4712. "
        "Linhagem: SIDRA 4711 v/617 c3/59998.",
    "domicilios_nao_ocupados_qtd": "Domicilios particulares permanentes nao ocupados, "
        "soma de vagos e uso ocasional. Unidade: domicilios. "
        "Linhagem: SIDRA 4711 v/617 c3/60001.",
    "domicilios_vagos_qtd": "Domicilios nao ocupados classificados como vagos. "
        "Unidade: domicilios. Baixa probabilidade de contrato ativo. "
        "Linhagem: SIDRA 4711 v/617 c3/60002.",
    "domicilios_uso_ocasional_qtd": "Domicilios nao ocupados de uso ocasional, tipicos "
        "de veraneio e turismo. Unidade: domicilios. Alta probabilidade de contrato "
        "ativo mesmo sem morador permanente. Linhagem: SIDRA 4711 v/617 c3/60003.",
    "uso_ocasional_pct": "Participacao de imoveis de uso ocasional no total "
        "recenseado. Unidade: percentual. Dominio: 0 a 100. Linhagem: "
        "domicilios_uso_ocasional_qtd dividido por domicilios_total_qtd x 100.",
    "perfil_ocupacao": "Classificacao do municipio pelo peso do uso ocasional. "
        "Dominio: Ocupacao permanente (abaixo de 10 por cento), Ocupacao mista "
        "(10 a 29), Veraneio ou turismo (30 ou mais).",
    "pib_total_mil_brl": "PIB a precos correntes. Unidade: mil reais. "
        "Linhagem: SIDRA 5938 v/37.",
    "impostos_liquidos_mil_brl": "Impostos liquidos de subsidios sobre produtos. "
        "Unidade: mil reais. Linhagem: SIDRA 5938 v/543.",
    "vab_total_mil_brl": "Valor adicionado bruto total. Unidade: mil reais. "
        "Identidade contabil: pib_total = vab_total + impostos_liquidos. "
        "Linhagem: SIDRA 5938 v/498.",
    "vab_agropecuaria_mil_brl": "VAB da agropecuaria. Unidade: mil reais. "
        "Linhagem: SIDRA 5938 v/513.",
    "vab_industria_mil_brl": "VAB da industria. Unidade: mil reais. "
        "Linhagem: SIDRA 5938 v/517.",
    "vab_servicos_mil_brl": "VAB dos servicos, exclusive administracao publica. "
        "Unidade: mil reais. Linhagem: SIDRA 5938 v/6575.",
    "vab_administracao_mil_brl": "VAB da administracao, defesa, educacao e saude "
        "publicas. Unidade: mil reais. Linhagem: SIDRA 5938 v/525.",
    "setor_dominante": "Setor economico com maior VAB. Dominio: Agropecuaria, "
        "Industria, Servicos, Administracao publica.",
    "pib_per_capita_brl": "PIB por habitante. Unidade: reais. Linhagem: "
        "pib_total_mil_brl x 1000 dividido por populacao_residente_hab.",
    "porte_populacional": "Faixa de porte populacional. Dominio: Ate 5 mil, 5 a 20 "
        "mil, 20 a 100 mil, Acima de 100 mil.",
})

In [0]:
mun = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.municipios")
mun.printSchema()

display(exibir(
    mun.orderBy(F.col("uso_ocasional_pct").desc()).limit(25),
    numericos={"populacao_residente_hab": (0, ""),
               "domicilios_total_qtd": (0, ""),
               "domicilios_ocupados_qtd": (0, ""),
               "domicilios_uso_ocasional_qtd": (0, ""),
               "domicilios_vagos_qtd": (0, ""),
               "uso_ocasional_pct": (2, " %")},
    texto=["nome_municipio", "mesorregiao_nome", "porte_populacional",
           "perfil_ocupacao"]))

A lista acima é o teste da hipótese: se os municípios com maior participação de
imóveis de uso ocasional forem os mesmos que apresentavam penetração acima de 100%,
a causa da anomalia está identificada.

In [0]:
display(
    mun.groupBy("perfil_ocupacao")
       .agg(F.count("*").alias("municipios_qtd"),
            F.round(F.avg("uso_ocasional_pct"), 2).alias("uso_ocasional_medio_pct"))
       .orderBy(F.col("municipios_qtd").desc())
)

In [0]:
por_setor = (
    mun.groupBy("setor_dominante")
       .agg(F.count("*").alias("municipios_qtd"),
            F.sum("populacao_residente_hab").alias("populacao_hab"),
            F.sum("domicilios_total_qtd").alias("domicilios_qtd"),
            F.round(F.avg("pib_per_capita_brl"), 2).alias("pib_per_capita_medio_brl"))
       .orderBy(F.col("municipios_qtd").desc())
)

display(exibir(por_setor,
    numericos={"municipios_qtd": (0, ""), "populacao_hab": (0, ""),
               "domicilios_qtd": (0, ""), "pib_per_capita_medio_brl": (2, "")},
    texto=["setor_dominante"]))

## 7. Qualidade de dados

As cinco dimensões exigidas pelo enunciado. O resultado é gravado em tabela, para
servir de evidência sem reexecução.

### 7.1 Completude

In [0]:
print("=== silver.acessos: nulos por coluna ===")
sa.select([F.sum(F.col(x).isNull().cast("int")).alias(x)
           for x in sa.columns if not x.startswith("_")]).show(vertical=True)

print("=== silver.municipios: nulos por coluna ===")
mun.select([F.sum(F.col(x).isNull().cast("int")).alias(x)
            for x in mun.columns if not x.startswith("_")]).show(vertical=True)

### 7.2 Unicidade

O grão declarado precisa ser único. Duplicata aqui indica erro de agregação e
inflaria todas as métricas da camada Gold.

In [0]:
total_linhas = sa.count()
combinacoes  = sa.select(*GRAO_ACESSOS).distinct().count()

print(f"silver.acessos: {total_linhas:,} linhas | {combinacoes:,} combinacoes da chave")
print(f"Grao integro: {total_linhas == combinacoes}")

for tabela, chave in [("empresas", ["cnpj"]),
                      ("tecnologias", ["tecnologia", "meio_acesso"]),
                      ("municipios", ["codigo_ibge"])]:
    d = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.{tabela}")
    n, u = d.count(), d.select(*chave).distinct().count()
    print(f"{tabela:<12} linhas={n:>5} | chaves distintas={u:>5} | integro={n == u}")

### 7.3 Consistência

Três verificações estruturais:

1. **Cobertura de municípios** entre Anatel e IBGE
2. **Identidade contábil do PIB** — `PIB = VAB total + impostos`
3. **Completude do VAB setorial** — a SIDRA publica o desdobramento por setor com
   defasagem maior que o PIB agregado
4. **Coerência entre as duas fontes de domicílios** — a categoria 59998 da tabela
   4711 deve reproduzir a variável 381 da tabela 4712

In [0]:
print("=== Completude do VAB setorial ===")
vab_nulos = mun.select(
    F.sum(F.col("vab_total_mil_brl").isNull().cast("int")).alias("vab_total"),
    F.sum(F.col("vab_agropecuaria_mil_brl").isNull().cast("int")).alias("agropecuaria"),
    F.sum(F.col("vab_industria_mil_brl").isNull().cast("int")).alias("industria"),
    F.sum(F.col("vab_servicos_mil_brl").isNull().cast("int")).alias("servicos"),
    F.sum(F.col("vab_administracao_mil_brl").isNull().cast("int")).alias("administracao"),
    F.sum(F.col("setor_dominante").isNull().cast("int")).alias("setor_dominante"),
)
vab_nulos.show()
print("Nulos aqui indicam que o ano de referencia do PIB nao publica o "
      "desdobramento setorial. Ver nota no notebook 01.\n")

print("=== Cobertura de municipios ===")
mun_anatel = sa.select("codigo_ibge").distinct()
mun_ibge   = mun.select("codigo_ibge").distinct()

print(f"Na Anatel        : {mun_anatel.count()}")
print(f"No IBGE          : {mun_ibge.count()}  (esperado {QTD_MUNICIPIOS_RS})")
print(f"Anatel sem IBGE  : {mun_anatel.join(mun_ibge, 'codigo_ibge', 'left_anti').count()}")
print(f"IBGE sem Anatel  : {mun_ibge.join(mun_anatel, 'codigo_ibge', 'left_anti').count()}")

print("\n=== Identidade contabil: PIB = VAB total + impostos ===")
divergencia_pib = (
    mun.withColumn("diferenca_mil_brl",
        F.abs(F.col("pib_total_mil_brl")
              - (F.col("vab_total_mil_brl") + F.col("impostos_liquidos_mil_brl"))))
       .filter(F.col("diferenca_mil_brl") > 1)
)
print(f"Municipios com divergencia acima de 1 mil reais: {divergencia_pib.count()}")

print("\n=== Coerencia entre SIDRA 4712 v/381 e SIDRA 4711 c3/59998 ===")
divergencia_dom = (
    mun.withColumn("diferenca_qtd",
        F.abs(F.col("domicilios_ocupados_qtd") - F.col("domicilios_ocupados_esp_qtd")))
       .filter(F.col("diferenca_qtd") > 0)
)
print(f"Municipios com divergencia: {divergencia_dom.count()}")
if divergencia_dom.count() > 0:
    display(divergencia_dom.select("nome_municipio", "domicilios_ocupados_qtd",
                                   "domicilios_ocupados_esp_qtd", "diferenca_qtd").limit(10))

print("\n=== Soma das especies bate com o total? ===")
soma_especies = (
    mun.withColumn("soma_partes",
        F.col("domicilios_ocupados_esp_qtd") + F.col("domicilios_nao_ocupados_qtd"))
       .withColumn("diferenca", F.col("domicilios_total_qtd") - F.col("soma_partes"))
)
soma_especies.select(
    F.round(F.avg("diferenca"), 2).alias("diferenca_media"),
    F.max("diferenca").alias("diferenca_maxima"),
).show()
print("Diferenca esperada acima de zero: o total inclui particular improvisado e "
      "domicilios coletivos, fora das duas categorias somadas.")

### 7.4 Acurácia

Compara a métrica calculada com a densidade oficial publicada pela Anatel — a única
validação possível contra fonte externa independente. A densidade da Anatel é por
100 habitantes.

In [0]:
densidade_anatel = (
    spark.table(f"{CATALOGO_BRONZE}.{SCHEMA}.anatel_densidade_raw")
    .filter((F.col("ano") == ANO_REF) & (F.col("mes") == MES_REF) &
            (F.col("nivel_geografico_densidade") == "Municipio"))
    .select(F.col("codigo_ibge"),
            limpar_numero("densidade", decimal_virgula=True)
                .cast(DoubleType()).alias("densidade_anatel"))
)

densidade_calc = (
    sa.groupBy("codigo_ibge").agg(F.sum("acessos_qtd").alias("acessos_qtd"))
      .join(mun.select("codigo_ibge", "nome_municipio", "populacao_residente_hab"),
            "codigo_ibge")
      .withColumn("densidade_calculada",
          F.round(100 * F.col("acessos_qtd") / F.col("populacao_residente_hab"), 6))
)

comparacao = (
    densidade_calc.join(densidade_anatel, "codigo_ibge", "inner")
    .withColumn("diferenca_abs",
        F.round(F.abs(F.col("densidade_calculada") - F.col("densidade_anatel")), 4))
)

print(f"Municipios comparados: {comparacao.count()}")
comparacao.select(
    F.round(F.mean("diferenca_abs"), 4).alias("diferenca_media"),
    F.round(F.max("diferenca_abs"), 4).alias("diferenca_maxima"),
    F.sum((F.col("diferenca_abs") < 0.01).cast("int")).alias("municipios_conformes"),
).show()

display(exibir(comparacao.orderBy(F.col("diferenca_abs").desc()).limit(10),
    numericos={"acessos_qtd": (0, ""), "populacao_residente_hab": (0, ""),
               "densidade_calculada": (4, ""), "densidade_anatel": (4, ""),
               "diferenca_abs": (4, "")},
    texto=["codigo_ibge", "nome_municipio"]))

### 7.5 Outliers

Compara a penetração sobre os dois denominadores. A diferença entre elas mede
exatamente o efeito dos imóveis não ocupados.

In [0]:
penetracao = (
    sa.groupBy("codigo_ibge").agg(F.sum("acessos_qtd").alias("acessos_total_qtd"))
      .join(mun.select("codigo_ibge", "nome_municipio", "mesorregiao_nome",
                       "perfil_ocupacao", "uso_ocasional_pct",
                       "domicilios_ocupados_qtd", "domicilios_total_qtd"),
            "codigo_ibge")
      .withColumn("penetracao_ocupados_pct",
          F.round(100 * F.col("acessos_total_qtd") / F.col("domicilios_ocupados_qtd"), 2))
      .withColumn("penetracao_enderecavel_pct",
          F.round(100 * F.col("acessos_total_qtd") / F.col("domicilios_total_qtd"), 2))
)

print("=== Distribuicao das duas penetracoes (%) ===")
penetracao.select(
    F.round(F.expr("percentile_approx(penetracao_ocupados_pct, 0.50)"), 2).alias("mediana_ocupados"),
    F.round(F.max("penetracao_ocupados_pct"), 2).alias("maximo_ocupados"),
    F.round(F.expr("percentile_approx(penetracao_enderecavel_pct, 0.50)"), 2).alias("mediana_enderecavel"),
    F.round(F.max("penetracao_enderecavel_pct"), 2).alias("maximo_enderecavel"),
).show()

acima_ocupados    = penetracao.filter(F.col("penetracao_ocupados_pct") > 100)
acima_enderecavel = penetracao.filter(F.col("penetracao_enderecavel_pct") > 100)

print(f"Acima de 100% sobre domicilios OCUPADOS      : {acima_ocupados.count()} de {penetracao.count()}")
print(f"Acima de 100% sobre domicilios RECENSEADOS   : {acima_enderecavel.count()} de {penetracao.count()}")

display(exibir(penetracao.orderBy(F.col("penetracao_ocupados_pct").desc()).limit(25),
    numericos={"acessos_total_qtd": (0, ""),
               "domicilios_ocupados_qtd": (0, ""),
               "domicilios_total_qtd": (0, ""),
               "uso_ocasional_pct": (2, " %"),
               "penetracao_ocupados_pct": (2, " %"),
               "penetracao_enderecavel_pct": (2, " %")},
    texto=["nome_municipio", "mesorregiao_nome", "perfil_ocupacao"]))

A comparação acima é a evidência central desta seção: se a troca de denominador
reduzir drasticamente o número de municípios acima de 100%, a hipótese dos imóveis de
uso ocasional está confirmada e o problema deixa de ser anomalia para virar escolha
metodológica documentada.

Os casos que permanecerem acima de 100% mesmo sobre o total recenseado exigem outra
explicação — múltiplos contratos por imóvel, ou acessos declarados no município da
prestadora.

### 7.6 Registro consolidado

In [0]:
diferenca_media = comparacao.select(F.round(F.mean("diferenca_abs"), 6)).first()[0]

verificacoes = [
    Row(dimensao="Completude",   verificacao="Nulos em silver.acessos",
        resultado=str(sa.filter(F.col("acessos_qtd").isNull()).count()), unidade="linhas"),
    Row(dimensao="Completude",   verificacao="Nulos em domicilios_total_qtd",
        resultado=str(mun.filter(F.col("domicilios_total_qtd").isNull()).count()), unidade="municipios"),
    Row(dimensao="Unicidade",    verificacao="Grao unico em silver.acessos",
        resultado=str(total_linhas == combinacoes), unidade="booleano"),
    Row(dimensao="Consistencia", verificacao="Municipios no IBGE (esperado 497)",
        resultado=str(mun.count()), unidade="municipios"),
    Row(dimensao="Consistencia", verificacao="Municipios Anatel sem par no IBGE",
        resultado=str(mun_anatel.join(mun_ibge, "codigo_ibge", "left_anti").count()), unidade="municipios"),
    Row(dimensao="Consistencia", verificacao="Divergencia na identidade do PIB",
        resultado=str(divergencia_pib.count()), unidade="municipios"),
    Row(dimensao="Consistencia", verificacao="Divergencia entre SIDRA 4712 e 4711 (ocupados)",
        resultado=str(divergencia_dom.count()), unidade="municipios"),
    Row(dimensao="Acuracia",     verificacao="Diferenca media vs densidade Anatel",
        resultado=str(diferenca_media), unidade="pontos de densidade"),
    Row(dimensao="Outliers",     verificacao="Penetracao acima de 100 pct sobre ocupados",
        resultado=str(acima_ocupados.count()), unidade="municipios"),
    Row(dimensao="Outliers",     verificacao="Penetracao acima de 100 pct sobre recenseados",
        resultado=str(acima_enderecavel.count()), unidade="municipios"),
]

gravar_silver(spark.createDataFrame(verificacoes), "qualidade_dados",
    "Camada Silver. Registro consolidado das verificacoes de qualidade executadas "
    "sobre as tabelas Silver, nas cinco dimensoes exigidas: completude, "
    "consistencia, unicidade, acuracia e outliers.")

display(spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.qualidade_dados"))

## 8. Inventário da camada

In [0]:
print(f"{'TABELA':<20} {'LINHAS':>10} {'COLUNAS':>9}")
print("-" * 41)
for t in sorted(spark.sql(f"SHOW TABLES IN {CATALOGO_SILVER}.{SCHEMA}").collect(),
                key=lambda x: x.tableName):
    d = spark.table(f"{CATALOGO_SILVER}.{SCHEMA}.{t.tableName}")
    print(f"{t.tableName:<20} {d.count():>10,} {len(d.columns):>9}")

---
**Camada Silver concluída.**

Próxima etapa: `03_gold_modelagem.ipynb` — modelo estrela e métricas de negócio.